# Module 3 — Dual-Pathway Hybrid Retrieval

**Goal:** given a clinician's query, search both pathways at once:
- **Vector path:** FAISS similarity search over Module 1's embedded chunks
- **Graph path:** Neo4j Cypher traversal over Module 2's UMLS-linked concept graph

Then merge both ranked lists using **Reciprocal Rank Fusion (RRF)**, prioritizing patients confirmed by the knowledge graph over those found only by text similarity, per the proposal's Section 7.5.

This is the Colab port of the local version. Two things needed correcting beyond paths and connection details:
1. The local model path was a hardcoded Windows filesystem path (`F:\Desktop\...`), which does not exist on Colab — this now loads Bio-ClinicalBERT from Hugging Face directly, the same way Module 1 does.
2. The query embedding function used plain mean pooling over all tokens, including padding. Module 1's Colab version was corrected to attention-masked mean pooling. If the query here used the old, uncorrected pooling while the stored corpus vectors use the corrected one, similarity search would be comparing embeddings computed two different ways — this silently degrades retrieval quality even though nothing throws an error. Query and corpus embeddings now use identical pooling.

Requires Module 1 and Module 2 to have been run already, since this reads their outputs directly.

## 1. Install packages

This notebook is a fresh Colab environment, so packages from both Module 1 and Module 2 need reinstalling here.

In [1]:
!pip install -q faiss-cpu transformers neo4j
!pip install -q nmslib-metabrainz==2.1.3
!pip install -q --no-deps scispacy
!pip install -q conllu pysbd scikit-learn scipy joblib

In [2]:
!pip install -q --no-deps https://s3-us-west-2.amazonaws.com/ai2-s2-scispacy/releases/v0.5.4/en_core_sci_sm-0.5.4.tar.gz

  Preparing metadata (setup.py) ... done


## 2. Imports, Drive mount, model config patch, and Neo4j connection

In [3]:
import os
import glob
import re
import pandas as pd
import numpy as np
import faiss
import torch
import en_core_sci_sm
from transformers import AutoTokenizer, AutoModel
from neo4j import GraphDatabase
from google.colab import drive, userdata

drive.mount('/content/drive')

# --- Patch the scispaCy model config before loading (same fix as Module 2) ---
pkg_dir = os.path.dirname(en_core_sci_sm.__file__)
for path in glob.glob(os.path.join(pkg_dir, "**", "config.cfg"), recursive=True):
    with open(path, "r") as f:
        content = f.read()
    fixed = re.sub(r'=\s*"True"', "= true", content)
    fixed = re.sub(r'=\s*"False"', "= false", fixed)
    if fixed != content:
        with open(path, "w") as f:
            f.write(fixed)

# --- Neo4j Aura connection ---
NEO4J_URI = userdata.get('NEO4J_URI')
NEO4J_USERNAME = userdata.get('NEO4J_USERNAME')
NEO4J_PASSWORD = userdata.get('NEO4J_PASSWORD')
driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USERNAME, NEO4J_PASSWORD))

# --- Paths from Module 1 ---
PROJECT_ROOT = "/content/drive/MyDrive/ClinicalTrust"
PROCESSED_DIR = os.path.join(PROJECT_ROOT, "data/processed")
MODEL_NAME = "emilyalsentzer/Bio_ClinicalBERT"

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)
print("Connections configured.")

/usr/local/lib/python3.13/dist-packages/spacy/util.py:971: UserWarning: [W095] Model 'en_core_sci_sm' (0.5.4) was trained with spaCy v3.7.4 and may not be 100% compatible with the current version (3.8.15). If you see errors or degraded performance, download a newer compatible model or retrain your custom model with the current spaCy version. For more details and available updates, run: python -m spacy validate
  warnings.warn(warn_msg)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Device: cuda
Connections configured.


## 3. Load Module 1's FAISS index + chunk metadata

In [4]:
faiss_index = faiss.read_index(os.path.join(PROCESSED_DIR, "pmc_patients.index"))
chunks_df = pd.read_parquet(os.path.join(PROCESSED_DIR, "chunks_metadata.parquet"))

print(f"FAISS index loaded: {faiss_index.ntotal} vectors")
print(f"Chunk metadata loaded: {chunks_df.shape}")
chunks_df.head()

FAISS index loaded: 10531 vectors
Chunk metadata loaded: (10531, 5)


,patient_id,chunk_id,age,gender,text
0,0,0_0,"[[60.0, 'year']]",M,this 60 - year - old male was hospitalized due...
1,1,1_0,"[[39.0, 'year']]",M,a 39 - year - old man was hospitalized due to ...
2,2,2_0,"[[57.0, 'year']]",M,one week after a positive covid - 19 result th...
3,3,3_0,"[[69.0, 'year']]",M,this 69 - year - old male was admitted to the ...
4,4,4_0,"[[57.0, 'year']]",M,this 57 - year - old male was admitted to the ...


## 4. Load Bio-ClinicalBERT (for embedding the query) and scispaCy (for extracting query entities)

The pooling logic in `embed_query` below is the attention-masked mean, matching Module 1's corrected `embed_texts` exactly — padding tokens are excluded from the average rather than included in it. Keeping this identical to Module 1 is what makes the FAISS distance comparison between query and corpus vectors meaningful.

In [5]:
from scispacy.linking import EntityLinker

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
bert_model = AutoModel.from_pretrained(MODEL_NAME).to(device)
bert_model.eval()

def embed_query(text):
    inputs = tokenizer([text], padding=True, truncation=True, max_length=512, return_tensors="pt").to(device)
    with torch.no_grad():
        outputs = bert_model(**inputs)

    mask = inputs["attention_mask"].unsqueeze(-1)
    summed = torch.sum(outputs.last_hidden_state * mask, dim=1)
    counts = torch.clamp(mask.sum(dim=1), min=1e-9)
    mean_pooled = summed / counts

    return mean_pooled.cpu().numpy()

nlp = en_core_sci_sm.load()
nlp.add_pipe("scispacy_linker", config={"resolve_abbreviations": True, "linker_name": "umls"})
linker = nlp.get_pipe("scispacy_linker")

print("Models loaded.")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: emilyalsentzer/Bio_ClinicalBERT
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
/usr/local/lib/python3.13/dist-packages/spacy/util.py:971: UserWarning: [W095] Model 'en_core_sci_sm' (0.5.4) was trained with spaCy v3.7.4 and may not be 100% compatible with the curr

Models loaded.


## 5. Vector pathway — FAISS search

Returns the top-k most similar chunks, aggregated to patient level. A patient may have multiple matching chunks; only their best (lowest) rank is kept.

In [6]:
def vector_search(query, k=20):
    query_vec = embed_query(query).astype("float32")
    distances, indices = faiss_index.search(query_vec, k)

    results = []
    for rank, idx in enumerate(indices[0]):
        row = chunks_df.iloc[idx]
        results.append({
            "patient_id": row["patient_id"],
            "rank": rank + 1,
            "distance": float(distances[0][rank]),
            "text_snippet": row["text"][:200]
        })

    df = pd.DataFrame(results)
    df = df.sort_values("rank").drop_duplicates(subset="patient_id", keep="first")
    return df.reset_index(drop=True)

## 6. Graph pathway — Cypher traversal

Extracts UMLS concepts from the query text, then finds patients whose graph mentions those same concepts, ranked by how many matching concepts each patient has.

In [7]:
def extract_query_cuis(query):
    doc = nlp(query)
    cuis = []
    for ent in doc.ents:
        if ent._.kb_ents:
            cui, score = ent._.kb_ents[0]
            cuis.append(cui)
    return list(set(cuis))

def graph_search(query, k=20):
    cuis = extract_query_cuis(query)
    if not cuis:
        return pd.DataFrame(columns=["patient_id", "rank", "matched_concepts"])

    with driver.session() as session:
        result = session.run(
            """
            MATCH (c:Concept)<-[:MENTIONS]-(p:Patient)
            WHERE c.cui IN $cuis
            RETURN p.patient_id AS patient_id, count(DISTINCT c) AS matched_concepts
            ORDER BY matched_concepts DESC
            LIMIT $k
            """,
            cuis=cuis, k=k
        )
        records = [dict(r) for r in result]

    df = pd.DataFrame(records)
    if not df.empty:
        df["rank"] = range(1, len(df) + 1)
    return df

## 7. Reciprocal Rank Fusion (RRF)

Combines both ranked lists into one, giving priority to knowledge-graph-confirmed results — the core retrieval innovation described in the proposal (Section 7.5).

$$RRF(patient) = \sum \frac{1}{k + rank}$$

A higher weight is applied to the graph pathway to reflect its higher epistemic reliability (typed relationships vs. text similarity).

In [8]:
def reciprocal_rank_fusion(vector_df, graph_df, rrf_k=60, graph_weight=1.5):
    scores = {}

    for _, row in vector_df.iterrows():
        pid = row["patient_id"]
        scores[pid] = scores.get(pid, 0) + 1 / (rrf_k + row["rank"])

    for _, row in graph_df.iterrows():
        pid = row["patient_id"]
        scores[pid] = scores.get(pid, 0) + graph_weight * (1 / (rrf_k + row["rank"]))

    fused = pd.DataFrame(list(scores.items()), columns=["patient_id", "rrf_score"])
    fused = fused.sort_values("rrf_score", ascending=False).reset_index(drop=True)
    fused["final_rank"] = range(1, len(fused) + 1)
    return fused

## 8. Put it all together — dual-pathway retrieval function

In [9]:
def dual_pathway_retrieve(query, k=20, top_n=10):
    vector_results = vector_search(query, k=k)
    graph_results = graph_search(query, k=k)
    fused = reciprocal_rank_fusion(vector_results, graph_results)

    print(f"Vector pathway matched {len(vector_results)} patients")
    print(f"Graph pathway matched {len(graph_results)} patients")
    print(f"Fused ranking (top {top_n}):\n")

    return fused.head(top_n), vector_results, graph_results

## 9. Test it

A clinical query is run through both pathways so the fused result and each pathway's individual contribution can be inspected side by side.

In [10]:
query = "patient with COVID-19 and respiratory distress"

fused_results, vector_results, graph_results = dual_pathway_retrieve(query)

print("--- Fused (final) ranking ---")
display(fused_results)

print("\n--- Vector pathway detail ---")
display(vector_results)

print("\n--- Graph pathway detail ---")
display(graph_results)

driver.close()

Vector pathway matched 20 patients
Graph pathway matched 20 patients
Fused ranking (top 10):

--- Fused (final) ranking ---


,patient_id,rrf_score,final_rank
0,32,0.037520,1
1,4,0.024590,2
2,11,0.024194,3
3,0,0.023810,4
4,17,0.023438,5
5,7,0.023077,6
6,3,0.022727,7
7,2,0.022388,8
8,25,0.022059,9
9,5,0.021739,10



--- Vector pathway detail ---


,patient_id,rank,distance,text_snippet
0,32,1,30.429550,( bipap ) with 40 l per minute of supplemental...
1,2201,2,31.311890,course. the patient unfortunately expired 12 d...
2,3732,3,31.705078,clinical course contrasted with the positivity...
3,4508,4,31.753632,"case 6 : 67 - year - old female, stanford type..."
4,1672,5,31.825821,##rrelating with diagnosis of myasthenia gravi...
5,1039,6,32.202835,"##osfamide ), and etoposide during march of 20..."
6,3230,7,32.753189,##rosis and polymorphonuclear cells were visib...
7,331,8,32.832428,##iothoracic surgery with an on - x mechanical...
8,3875,9,32.850189,sigmoid colon. an autopsy specimen of the ulce...
9,381,10,32.906662,but she had igg antibodies to covid - and met ...



--- Graph pathway detail ---


,patient_id,matched_concepts,rank
0,4,2,1
1,11,2,2
2,0,2,3
3,17,2,4
4,7,2,5
5,3,2,6
6,2,2,7
7,25,2,8
8,5,2,9
9,26,2,10


## Next steps

1. Try a few more test queries covering different conditions present in the current patient subset (Module 2's `linked_df["canonical_name"]` is a good source of concepts actually present in the graph).
2. Once this works on the small subset, scale Modules 1 and 2 together to a larger `SUBSET_SIZE` so retrieval has a richer pool to search.
3. Module 4 takes these fused, ranked patient results as the retrieved context passed into the reasoning agent (`deepseek-r1:7b` on Colab, pending validation before any 32b run).
4. Save this notebook into `ClinicalTrust/notebooks/` on Drive alongside Modules 1 and 2.